# RAG-Only Baseline And Failure Modes

This notebook builds a local retriever and then uses a real provider call for grounded answer generation. RAG adds evidence to the prompt; it does not train model weights.


## Retrieval Failure Modes

A RAG system can fail before the model ever writes an answer. The retriever may miss the right document, return a weakly related document, or rank the correct source below irrelevant context. The generator can also fail after retrieval by ignoring evidence, dropping citations, or answering beyond the context.

For this reason, retrieval evaluation and generated-answer evaluation should stay separate. Recall@k tells us whether the expected document appears in the top results. Citation and abstention checks tell us whether the generation path respects the evidence that was retrieved.


## Retrieval Terms In Plain Language

A **query** is the text used to search the corpus. It can be the user question directly, or a rewritten version of it. A **chunk** is a smaller piece of a document that can be retrieved independently. A **score** estimates how similar or relevant a document is to the query. A **top-k result** is one of the first `k` records returned by the retriever.

The notebook uses simple token counts instead of embeddings so the mechanics stay visible. In production, the same concepts usually appear with embedding models and vector databases: query embedding, chunk embedding, similarity score, and top-k retrieval. The teaching point is the same: generation quality depends on whether the right evidence enters the prompt.


### Exercise

If the retriever misses the correct document, should the generator still answer confidently?


### Solution

No. It should abstain or ask for more context because the answer would not be grounded in available evidence.


## Visual Map: Retrieval Quality Comes Before Generation

RAG quality depends on whether the right evidence reaches the model. A fluent generated answer is still weak if retrieval missed the supporting passage.

**Semantic-search pipeline.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/semantic_search_pipeline.png). This visual shows the retrieval half of the system: encode the query, search indexed passages, then pass selected evidence downstream.

<img src="assets/hf-semantic-search-pipeline.png" alt="Semantic search pipeline" style="max-width: 760px; width: 100%; height: auto;">

---

After the pipeline view, the next visual focuses on tuning retrieval settings and measuring their impact.

**RAG settings and accuracy trade-off.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/RAG_settings_accuracy.png). This visual is useful because retrieval settings are measurable choices, not decoration around the LLM call.

<img src="assets/hf-cookbook-rag-settings-accuracy.png" alt="RAG settings and accuracy" style="max-width: 760px; width: 100%; height: auto;">

---

The last visual adds an operational concern: repeated or similar requests should not always require the full retrieval-and-generation path.

**Semantic cache architecture.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/semantic_cache.jpg). The cache example makes a practical point: once retrieval and generation are real API paths, latency and repeated-query behavior become part of the architecture discussion.

<img src="assets/hf-cookbook-semantic-cache.jpg" alt="Semantic cache architecture" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

In a RAG pipeline, should retrieval quality be checked before or after judging the generated answer?


### Solution

Check retrieval quality first. If the right evidence is missing, a fluent generated answer is not trustworthy.


## Load Dependencies And Shared Data

The setup is split into small cells so each responsibility is visible: imports first, then `.env` loading and fixture loading.


In [ ]:
import json
import math
import os
import re
from collections import Counter
from pathlib import Path
from pprint import pprint
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

The notebooks share one small support-policy fixture. Keeping it shared makes the strategy comparisons fair: each notebook sees the same documents, requests, and style examples.


In [ ]:
# Load optional provider settings from .env without overwriting shell variables.
load_dotenv(dotenv_path=Path(".env"), override=False)

DATA_PATH = Path("data") / "strategy_cases.json"
with DATA_PATH.open(encoding="utf-8") as file:
    DATA = json.load(file)

# Keep each fixture role explicit so later cells do not mix evidence, examples, and evaluation.
DOCUMENTS = DATA["documents"]
REQUESTS = DATA["support_requests"]
STYLE_EXAMPLES = DATA["style_examples"]
SCENARIOS = DATA["strategy_scenarios"]

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, and {len(STYLE_EXAMPLES)} style examples."
)

## Configure The Provider

The notebooks use the OpenAI-compatible client style. Groq is tried first by default; Mistral is the fallback when the preferred provider has no key.


In [ ]:
class ProviderConfig(BaseModel):
    """Provider settings for one OpenAI-compatible chat endpoint."""

    provider: str
    model: str
    base_url: str
    api_key: str | None = None
    dry_run: bool = True

Provider selection is intentionally explicit. This makes fallback behavior easy to inspect before any model call happens.


In [ ]:
def provider_order(preferred_provider: str | None = None) -> list[str]:
    """Return the preferred provider followed by the other supported provider."""
    requested = (preferred_provider or os.getenv("LLM_PROVIDER") or "groq").lower()

    if requested == "groq":
        return ["groq", "mistral"]
    if requested == "mistral":
        return ["mistral", "groq"]

    # Unknown values keep dry mode readable instead of failing during setup.
    return [requested]

The loader returns dry mode when no key is available. Dry mode is part of the teaching design: notebooks stay runnable during review and in classrooms without credentials.


Each provider has a tiny environment-to-config adapter. Keeping this separate makes the fallback loader shorter and easier to read.


In [ ]:
def provider_config_from_env(provider_name: str) -> ProviderConfig | None:
    """Return live provider settings when that provider has an API key."""
    if provider_name == "groq" and os.getenv("GROQ_API_KEY"):
        return ProviderConfig(
            provider="groq",
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY"),
            dry_run=False,
        )

    if provider_name == "mistral" and os.getenv("MISTRAL_API_KEY"):
        return ProviderConfig(
            provider="mistral",
            model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
            base_url="https://api.mistral.ai/v1",
            api_key=os.getenv("MISTRAL_API_KEY"),
            dry_run=False,
        )

    return None

The loader checks providers in order and falls back to dry mode when no key is available.


In [ ]:
def load_provider_config(preferred_provider: str | None = None) -> ProviderConfig:
    """Load live provider settings, or return a deterministic dry-run config."""
    # Check providers in priority order so fallback behavior is deterministic.
    for provider_name in provider_order(preferred_provider):
        config = provider_config_from_env(provider_name)
        if config is not None:
            return config

    # No key was found, so keep the notebook runnable without network calls.
    fallback_model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
    return ProviderConfig(
        provider=provider_order(preferred_provider)[0],
        model=fallback_model,
        base_url="dry-run",
    )

In [ ]:
CONFIG = load_provider_config()
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

if CONFIG.dry_run:
    print("Set GROQ_API_KEY or MISTRAL_API_KEY in .env to run live model calls.")

## Call The Model Or Use Dry Mode

The next cells isolate client creation, request assembly, and fallback behavior. That keeps provider plumbing separate from the lesson logic.


In [ ]:
def get_client(config: ProviderConfig) -> OpenAI | None:
    """Create a client only when a live API key is configured."""
    if config.dry_run or config.api_key is None:
        return None

    return OpenAI(api_key=config.api_key, base_url=config.base_url)

Request parameters are built in one small helper so response formatting can be added only where a notebook needs structured JSON.


In [ ]:
def completion_kwargs(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> dict[str, Any]:
    """Build provider request arguments for chat completions."""
    kwargs: dict[str, Any] = {
        "model": CONFIG.model,
        "messages": messages,
        "temperature": temperature,
    }

    # JSON mode is optional because not every provider/model combination needs it.
    if response_format is not None:
        kwargs["response_format"] = response_format

    return kwargs

`call_model` is the single gateway used by lesson cells. If live credentials are missing or a live call fails, the notebook returns a deterministic teaching response.


Live completion execution is separated from dry-mode routing so the failure handling is easy to follow.


In [ ]:
def live_completion_content(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> str:
    """Run the live chat completion and return message content."""
    client = get_client(CONFIG)
    assert client is not None

    completion = client.chat.completions.create(
        **completion_kwargs(
            messages, temperature=temperature, response_format=response_format
        )
    )
    return completion.choices[0].message.content or ""

`call_model` now only decides between dry output, live output, and live-call fallback.


In [ ]:
def call_model(
    messages: list[dict[str, str]],
    *,
    dry_response: str,
    temperature: float = 0.2,
    response_format: dict[str, str] | None = None,
) -> str:
    """Call the configured provider, or return deterministic dry output."""
    # Dry mode is the expected path for participants without provider credentials.
    if CONFIG.dry_run:
        return dry_response

    try:
        return live_completion_content(
            messages, temperature=temperature, response_format=response_format
        )
    except Exception as error:
        print(f"Live provider call failed: {error}")
        print("Using deterministic dry response so the notebook remains runnable.")
        return dry_response

## Build A Transparent Local Retriever

The retriever is deliberately simple: tokenize text, build term-frequency counters, score cosine similarity, then sort candidate documents.


In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase text and keep word-like tokens for a tiny local retriever."""
    return re.findall(r"[a-z0-9]+", text.lower())


def term_counts(text: str) -> Counter[str]:
    """Count token frequency so repeated domain terms increase similarity."""
    return Counter(tokenize(text))

Document vectors are computed once. In a production RAG stack this role is usually handled by an embedding model and vector database; here it stays inspectable.


In [ ]:
DOCUMENT_VECTORS = {
    document["doc_id"]: term_counts(document["title"] + " " + document["text"])
    for document in DOCUMENTS
}

In [ ]:
# Exercise: count how many documents were indexed.
# indexed_document_count = ...


In [ ]:
# Solution
indexed_document_count = len(DOCUMENT_VECTORS)
print(indexed_document_count)

Cosine similarity gives a compact relevance score between the query counter and each document counter. The search function keeps source metadata attached for later citations.


Cosine similarity is the scoring function for this transparent retriever.


In [ ]:
def cosine(left: Counter[str], right: Counter[str]) -> float:
    """Compute cosine similarity between sparse term-frequency counters."""
    shared_terms = set(left) & set(right)
    numerator = sum(left[term] * right[term] for term in shared_terms)
    left_norm = math.sqrt(sum(value * value for value in left.values()))
    right_norm = math.sqrt(sum(value * value for value in right.values()))

    # Empty vectors cannot be compared, so they receive no retrieval score.
    if left_norm == 0 or right_norm == 0:
        return 0.0

    return numerator / (left_norm * right_norm)

Search keeps score calculation separate from context formatting so each step can be inspected.


In [ ]:
def search_documents(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Return top documents with retrieval scores and citation metadata."""
    query_vector = term_counts(query)
    scored_documents = []

    for document in DOCUMENTS:
        # Keep the document fields and add a rounded score for inspection.
        score = cosine(query_vector, DOCUMENT_VECTORS[document["doc_id"]])
        scored_documents.append({**document, "score": round(score, 4)})

    return sorted(scored_documents, key=lambda item: item["score"], reverse=True)[
        :top_k
    ]

The generator receives compact context lines with explicit source IDs for citation checks.


In [ ]:
def format_context(rows: list[dict[str, Any]]) -> str:
    """Format retrieved rows as citeable prompt context."""
    return "\n".join(f"[{row['doc_id']}] {row['text']}" for row in rows)

Use `retrieve` as the public lesson name for document search. The implementation stays one line so participants can see that retrieval is just ranked search at this stage.


In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[dict[str, Any]]:
    """Lesson-friendly alias for the local document search helper."""
    return search_documents(query, top_k=top_k)

## Retrieve Policy Evidence

The retriever is deliberately transparent. It uses sparse token cosine similarity so participants can inspect failure modes before swapping in embeddings or a vector database.


In [ ]:
question = REQUESTS[1]["question"]
retrieved = retrieve(question, top_k=3)
pprint(retrieved)

In [ ]:
# Exercise: inspect the top retrieved document ID for this question.
# top_doc_id = ...


In [ ]:
# Solution
top_doc_id = retrieved[0]["doc_id"]
print(top_doc_id)

## Call The Model With Citations

The prompt requires the answer to cite source IDs. If retrieval is weak, the model should abstain rather than invent a policy.


The answer schema makes grounding inspectable: the output must say what it answered from and whether it abstained.


In [ ]:
class CitedAnswer(BaseModel):
    """Structured representation of a grounded answer."""

    answer: str
    source_ids: list[str]
    status: str

Retrieve first and abstain before calling the model when the best evidence is too weak.


In [ ]:
def retrieve_or_abstain(
    question: str, *, threshold: float = 0.08
) -> list[dict[str, Any]]:
    """Return evidence only when the top retrieval score is high enough."""
    evidence = retrieve(question, top_k=3)

    if not evidence or evidence[0]["score"] < threshold:
        return []

    return evidence

The RAG answer path builds a cited context prompt, calls the provider, and validates the JSON response.


Build the cited-answer prompt separately from the RAG control flow.


In [ ]:
def rag_messages(question: str, evidence: list[dict[str, Any]]) -> list[dict[str, str]]:
    """Create a context-grounded prompt for cited JSON output."""
    context = format_context(evidence)
    return [
        {
            "role": "system",
            "content": "Answer only from the context. Return JSON with answer, source_ids, and status.",
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

The dry response mirrors the expected cited JSON structure.


In [ ]:
def dry_cited_answer(evidence: list[dict[str, Any]]) -> str:
    """Create deterministic cited output for credential-free execution."""
    return json.dumps(
        {
            "answer": f"{evidence[0]['text']} [{evidence[0]['doc_id']}]",
            "source_ids": [evidence[0]["doc_id"]],
            "status": "answered",
        }
    )

The final answer helper now only handles abstention, provider routing, and validation.


In [ ]:
def rag_answer(question: str, *, threshold: float = 0.08) -> CitedAnswer:
    """Retrieve policy context and ask the provider for a cited answer."""
    evidence = retrieve_or_abstain(question, threshold=threshold)
    if not evidence:
        return CitedAnswer(
            answer="I do not know from the available policy documents.",
            source_ids=[],
            status="abstained",
        )

    raw = call_model(
        rag_messages(question, evidence),
        dry_response=dry_cited_answer(evidence),
        response_format={"type": "json_object"},
    )

    try:
        return CitedAnswer.model_validate_json(raw)
    except ValidationError:
        return CitedAnswer(answer=raw, source_ids=[], status="parse_failed")

Run the grounded path once so the cited response can be inspected before evaluating retrieval quality.


In [ ]:
answer = rag_answer(question)
pprint(answer.model_dump())

## Evaluate Retrieval Recall

Before judging generation, check whether retrieval found the expected source. If the right document is absent, the answer is already constrained by poor evidence.


## What Recall@k Does And Does Not Tell You

**Recall@k** asks a narrow question: did the expected source appear somewhere in the top `k` retrieved records? If `recall@1` is low but `recall@3` is high, the retriever often finds the right document but ranks it below other documents. That may still be usable if the generator receives all three records, but it increases the risk of distraction.

Recall does not judge whether the final answer is well written, complete, or safe. It also does not prove that the cited sentence supports every claim in the answer. That is why RAG evaluation usually has at least two layers: retrieval metrics first, then answer-grounding checks.


In [ ]:
def recall_at_k(k: int = 3) -> float:
    """Measure whether expected policy documents appear in top-k retrieval."""
    hits = 0
    for request in REQUESTS:
        retrieved_ids = [
            record["doc_id"] for record in retrieve(request["question"], top_k=k)
        ]
        hits += request["expected_doc"] in retrieved_ids
    return hits / len(REQUESTS)


for k in [1, 2, 3]:
    print(f"recall@{k} = {recall_at_k(k):.2f}")

### Exercise

If `recall@2` is higher than `recall@1`, what changed?


### Solution

The correct document was not always ranked first, but it appeared when the retriever was allowed to return two documents.


In [ ]:
unsupported_question = "Where should I park my bicycle near campus?"
pprint(rag_answer(unsupported_question, threshold=0.18).model_dump())